# Exercise 2 - Markov Chains

In this exercise, we will answer inference queries on a Markov chain, one of the simplest directed graphical models.

In the event of a persistent problem, do not hesitate to contact me at

- tobias.pretschold@uni-jena.de

### Submission

- Deadline of submission: Tue, 3 November 11:59 pm.
- Submission via Moodle

### Help
In case you cannot solve a task, you can use the saved values within the `help` directory. The checkpoints are named after the corresponding functions; the functions from Exercise 1 are in `help/exercise_01`.

- Load arrays with [Numpy](https://numpy.org/doc/stable/reference/generated/numpy.load.html)
```
np.load('help/array_name.npy')
```
- Load functions with [Dill](https://dill.readthedocs.io/en/latest/dill.html)
```
import dill
with open('help/some_func.pkl', 'rb') as f:
    func = dill.load(f)
```

to continue working on the other tasks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Markov Chains

In Exercise 1, we stored the joint distribution of 12 variables as a full tensor. This does not scale, since the tensor grows exponentially with the number of variables. To handle distributions of hundreds of variables (or even more), we have to make simplifying assumptions. These are either algebraic (the distribution factorizes) or conceptual (some variables are (conditionally) independent). As you might guess already, these two notions are intimately related.

Directed graphical models provide a convenient way to visualize the simplifying assumptions about the distribution we are making. Each variable $X_i$ is represented as a node in the graph. An edge going from $X_i$ to $X_j$ intuitively means "$X_i$ influences $X_j$". More precisely, if the variables are arranged in a directed acyclic graph (DAG), the joint distribution factorizes into the conditional distributions of the variables given their parents $\mathrm{pa}(i)$,

\begin{equation}
p(x) = \prod_i p(x_i \mid x_{\mathrm{pa}(i)})\,.
\end{equation}

These conditional distributions are stored as conditional probability tables (CPTs). Equivalently, every variable is conditionally independent of its non-descendants given its parents,

\begin{equation}
X_i \perp X_{\mathrm{nd}(i)} \mid X_{\mathrm{pa}(i)}\,,
\end{equation}

where $\mathrm{nd}(i)$ are the non-descendants of $X_i$ other than its parents. (Don't worry if this looks complicated at first sight. It probably is.)

A Markov chain now is a special case, where the variables form a sequence $X_0, X_1, \dots, X_{n-1}$, as shown in the picture. Every variable $X_{t+1}$ has only its predecessor $X_t$ as parent.

<div>
<img src="images/markov_chain.png" width="600"/>
</div>

Interpreting this graph as described above, the joint distribution factorizes in the following way:

\begin{equation}
p(x_0, \dots, x_{n-1}) = p(x_0) \prod_{t=0}^{n-2} p(x_{t+1} \mid x_t) = p(x_0) \prod_{t=0}^{n-2} T_t[x_t, x_{t+1}]\,,
\end{equation}

where the CPT $T_t$ is the matrix with entries $T_t[i, j] = p(X_{t+1} = j \mid X_t = i)$. Equivalently, the conditional independence statements

\begin{equation}
X_{t+1} \perp (X_0, \dots, X_{t-1}) \mid X_t
\end{equation}

hold for all $t$, i.e. given the present, the future is independent of the past.

For this lab, we use fake weather data: $X_t$ is the weather on day $t$ of a year, with the $K = 3$ values `sunny` (0), `cloudy` (1) and `rainy` (2). For the $n = 365$ days, the probability tensor would have $3^{365} \approx 10^{174}$ entries, the CPTs only have $3 + 364 \cdot 9 = 3279$.

### Task 1

Load $p(x_0)$ from `p0.npy` and the CPTs from `T.npy`, where `T[t]` holds $T_t$. Check that $p(x_0)$ and all rows of all CPTs are probability distributions.

In [ ]:
n, K = 365, 3
SUNNY, CLOUDY, RAINY = 0, 1, 2

# TODO: load p0 and T, check that they are probability distributions

### Task 2

For the first few variables, we can still afford the full tensor. Implement `joint_tensor`, which computes the probability tensor of the first $m$ variables,

\begin{equation}
P[x_0, \dots, x_{m-1}] = p(x_0) \prod_{t=0}^{m-2} T_t[x_t, x_{t+1}]\,.
\end{equation}

Then compute `P8` for $m = 8$. As a sanity check, `P8` should sum to 1.

In [ ]:
def joint_tensor(p0: np.ndarray, T: np.ndarray, m: int) -> np.ndarray:
    '''
    Computes the probability tensor of the first m variables of the chain.

    @Params:
        p0... distribution of the first variable
        T... CPTs, T[t][i, j] = p(X_{t+1} = j | X_t = i)
        m... number of variables

    @Returns:
        probability tensor with one axis per variable
    '''
    # TODO: implement
    pass

# TODO: compute P8

# Distributive Law

Summing over the full tensor costs $K^n$ operations. With the distributive law $ab + ac = a(b + c)$, we can push every sum inside the product, past all factors that do not depend on the summed variable. For the marginal $p(x_t)$:

- The variables right of $t$ are summed out starting with $x_{n-1}$. Each of these sums is 1, since every row of a CPT sums to 1.
- The variables left of $t$ are summed out starting with $x_0$:
\begin{equation}
p(x_t) = \sum_{x_{t-1}} \cdots \sum_{x_1} \Big( \sum_{x_0} p(x_0)\, T_0[x_0, x_1] \Big)\, T_1[x_1, x_2] \cdots T_{t-1}[x_{t-1}, x_t]\,.
\end{equation}

Each inner sum is a vector-matrix product $p(x_{t+1}) = \sum_{x_t} p(x_t)\, T_t[x_t, x_{t+1}]$, so all marginals together cost only about $n K^2$ operations.

### Task 3

Implement `chain_marginals`, which computes $p(x_t)$ for all $t$. Check the first 8 marginals against `P8`. The cell below then plots your marginals over the year.

As a sanity check, the probability of a sunny December 31, $p(X_{364} = \text{sunny})$, should be `0.20029599`.

In [ ]:
def chain_marginals(p0: np.ndarray, T: np.ndarray) -> np.ndarray:
    '''
    Computes the marginal distributions p(x_t) of all variables of the chain.

    @Params:
        p0... distribution of the first variable
        T... CPTs, T[t][i, j] = p(X_{t+1} = j | X_t = i)

    @Returns:
        array of shape (n, K), row t holds p(x_t)
    '''
    # TODO: implement
    pass

# TODO: compare to P8

In [ ]:
# plot the marginals over the year
M = chain_marginals(p0, T)
for k, label in enumerate(['sunny', 'cloudy', 'rainy']):
    plt.plot(M[:, k], label=label)
plt.xlabel('day $t$')
plt.ylabel('$p(x_t)$')
plt.legend()
plt.show()

## Conditionals

To get $p(x_t \mid x_B)$, we compute $p(x_t, x_B)$ for all values of $x_t$ and normalize. This is the same sum as for the marginal, except that the observed variables are fixed to their values instead of summed over. Since the factors left of $x_t$ and the factors right of $x_t$ only have $x_t$ in common, the sum splits into a left and a right part. For example, for $n = 4$ and the evidence $X_3 = \text{rainy}$,

\begin{equation}
p(x_1, X_3 = \text{rainy}) = \underbrace{\sum_{x_0} p(x_0)\, T_0[x_0, x_1]}_{\text{left part}} \cdot \underbrace{\sum_{x_2} T_1[x_1, x_2]\, T_2[x_2, \text{rainy}]}_{\text{right part}}\,.
\end{equation}

As in Task 3, we compute the left part step by step from the left. The right part we compute step by step from the right, starting with a vector of ones. Whenever we reach an observed variable, we fix it by setting all entries of the current vector to zero, except the one of the observed value.

### Task 4

Implement `chain_conditional` and check it against `P8` with your `conditional` from Exercise 1, e.g. for $p(x_3 \mid X_1 = \text{sunny}, X_6 = \text{rainy})$. The cell below then plots $p(X_t = \text{rainy} \mid X_{200} = \text{rainy})$ and $p(X_t = \text{rainy})$ for the days around day 200.

As a sanity check, $p(X_{195} = \text{rainy} \mid X_{200} = \text{rainy})$ should be `0.17057201`.

In [ ]:
def chain_conditional(p0: np.ndarray, T: np.ndarray, t: int, B: list, x_B: list) -> np.ndarray:
    '''
    Computes the conditional distribution p(x_t | x_B) of the variable t given the observed values x_B.

    @Params:
        p0... distribution of the first variable
        T... CPTs, T[t][i, j] = p(X_{t+1} = j | X_t = i)
        t... index of the query variable
        B... list of indices of the observed variables (evidence)
        x_B... list of observed values for the variables in B

    @Returns:
        probability vector of length K
    '''
    # TODO: implement
    pass

# TODO: compare to P8

In [ ]:
# plot p(X_t = rainy | X_200 = rainy) and p(X_t = rainy) around day 200
days = range(185, 216)
plt.plot(days, [chain_conditional(p0, T, t, [200], [RAINY])[RAINY] for t in days], 'o-',
         label='$p(X_t = $ rainy $\\mid X_{200} = $ rainy$)$')
plt.plot(days, M[days, RAINY], 'o-', label='$p(X_t = $ rainy$)$')
plt.xlabel('day $t$')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.show()